# Step 2: AWQ 量化（W4A16，llm-compressor 统一路径）

**目标**：用 `llmcompressor` 的 `AWQModifier` + `QuantizationModifier` 把 Qwen2.5-7B-Instruct 量化成 **W4A16（4-bit 权重 / 16-bit 激活）**——显存省到约 1/4，激活保持高精度（所以对长上下文 / 注意力类负载友好），产物同样是 compressed-tensors 格式。

**对应 OUTLINE 课时**：2.4 AWQ W4A16 全流程（~55 分钟）。

> AWQ（Activation-aware Weight Quantization）的核心思想：**不是所有权重都同等重要**——保护那些"大激活输入对应"的显著权重通道（通过 per-channel scale 把它们的量化误差缩小），就能在 4-bit 下几乎不掉点。与 FP8/SmoothQuant 不同，AWQ 只压权重、不压激活（W4A16）。

In [ ]:
%%capture
import subprocess, pathlib, json
import torch
import ipytest
ipytest.autoconfig()
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from llmcompressor.modifiers.transform.awq import AWQModifier

In [ ]:
# 解析仓库根目录（cwd 无关）：notebooks 通过 `uv run --directory envs/quant jupyter lab`
# 启动，但 jupyter 的 cwd 是所在 shell 的 cwd（不是 --directory 目标），所以
# 所有路径都从 git 仓库根派生，绝不依赖裸相对路径。
import subprocess, pathlib

REPO_ROOT = pathlib.Path(
    subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
)
MODEL_DIR = REPO_ROOT / "models" / "Qwen2.5-7B-Instruct"      # 与 download_model.sh 默认一致
TINY_MODEL_DIR = REPO_ROOT / "models" / "Qwen2.5-0.5B-Instruct"  # L3 先在 0.5B 上验，再上 7B
OUT_ROOT = REPO_ROOT / "out"                                  # 已 gitignore
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)
print("GPU OK" if __import__("torch").cuda.is_available() else "无 GPU（仅 L1/L2 可跑）")

## 原理：AWQ 的两段式 recipe

llmcompressor 的 AWQ 是**两段式 recipe**（与 SmoothQuant 的两段式结构对称）：

```python
recipe = [
    AWQModifier(),                                            # 第一段：搜每个 Linear 的 scale，平滑显著权重
    QuantizationModifier(targets="Linear", scheme="W4A16_ASYM", ignore=["lm_head"]),  # 第二段：把权重压成 4-bit
]
```

- `AWQModifier()`：在校准数据上前向，用网格搜索（grid search）为每个权重通道找一个 scale，使"大激活通道"的权重量化误差最小——**这就是 AWQ 的核心**（保护显著权重）。
- `QuantizationModifier(scheme="W4A16_ASYM")`：权重 4-bit **非对称**（ASYM，带 zero-point）、**group-wise**（默认 group_size=128，每 128 个权重共享一组 scale/zp）；激活**不量化**（A16）。
- **需要校准数据**（AWQ 要看激活分布）：用 `wikitext-2`，AWQ 极省样本，128–256 条即可（OUTLINE 2.2）。

**易错点（OUTLINE 标注）**：
- 旧 `from llmcompressor.modifiers.awq import AWQModifier` 已**废弃**（兼容 shim），当前路径是 `llmcompressor.modifiers.transform.awq`。
- AutoAWQ 的 `quant_config` **没有 `exclude_modules` 字段**——AWQ 排除层走加载侧（`AwqConfig.modules_to_not_convert`，默认 `['lm_head']`），量化期 AWQ 硬编码跳过 lm_head。课程用 llmcompressor 统一路径，靠 `ignore=["lm_head"]` 在量化期显式排除。

## 本步填空

1. **`build_awq_recipe(ignore, scheme, group_size)`** —— 构造 AWQ 两段式 recipe（AWQModifier + QuantizationModifier）。
2. **`build_calibration_dataset(tokenizer, n_samples, seq_len)`** —— 把 `wikitext-2` 文本 tokenize 成校准样本（教学：理解校准数据的形状——每条是定长 token id 序列）。
3. **`awq_config_summary(qc)`** —— 从产物 `quantization_config` 抽出 W4A16 关键字段（num_bits=4 / symmetric=False / group_size / 无 input_activations）。

In [ ]:
def build_awq_recipe(ignore=("lm_head",), scheme="W4A16_ASYM"):
    """返回 AWQ 两段式 recipe（list of modifiers）。

    要求：
      - 第一段：AWQModifier()（无参数，用默认 grid search 搜显著权重 scale）
      - 第二段：QuantizationModifier(targets="Linear", scheme=scheme, ignore=list(ignore))
    返回 [awq_modifier, quant_modifier]。

    注意：W4A16_ASYM 的 group_size=128 是 scheme 内置默认，无需显式传（QuantizationModifier
          没有 group_size 关键字参数；显式传会 ValidationError）。
    """
    # TODO: 构造并返回两个 modifier 组成的 list
    #       提示：AWQModifier() + QuantizationModifier(targets="Linear", scheme=scheme, ignore=list(ignore))
    raise NotImplementedError


# 脚手架（提供）：真正跑 AWQ 的 execution，调用你填好的 recipe + 校准数据
def run_awq_quantize(model, tokenizer, calib_texts, save_dir, n_samples=256, seq_len=2048):
    recipe = build_awq_recipe()
    oneshot(
        model=model, tokenizer=tokenizer,
        dataset=build_calibration_dataset(tokenizer, n_samples=n_samples, seq_len=seq_len,
                                          raw_texts=calib_texts),
        recipe=recipe, max_seq_length=seq_len, num_calibration_samples=n_samples,
    )
    save_dir = pathlib.Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_dir)
    return save_dir

In [ ]:
def build_calibration_dataset(tokenizer, n_samples=256, seq_len=2048, raw_texts=None):
    """构造 AWQ 校准数据集（datasets.Dataset，列名 "text"）。

    - 若 raw_texts 不为 None：直接包装成 Dataset({"text": [...]})（用于 tiny / 测试，免网络）。
    - 若 raw_texts 为 None：从 wikitext-2 加载（注意新版 datasets 要用命名空间 repo id
      `Salesforce/wikitext` + 配置名 `wikitext-2-raw-v1`，旧的裸 `wikitext` 会 HfUriError），
      shuffle(seed=42) 后取前 n_samples 条**非空**文本。

    返回：datasets.Dataset（含 "text" 列）。oneshot 内部会用 tokenizer 对 "text" 列
    做 tokenize + 按 max_seq_length 截断，所以这里返回原始文本列即可。
    """
    # TODO: 实现（提示：from datasets import load_dataset, Dataset）
    raise NotImplementedError

In [ ]:
def awq_config_summary(quantization_config):
    """从 W4A16 产物的 quantization_config 抽出关键字段。

    返回 dict，至少含：
      - "quant_method"      : str
      - "weights_num_bits"  : int   （4）
      - "weights_symmetric" : bool  （W4A16_ASYM -> False）
      - "weights_group_size": int   （128）
      - "weights_strategy"  : str   （"group"）
      - "has_input_activations": bool  （AWQ 不量化激活 -> False / None）
      - "targets"           : list
    """
    # TODO: 解析并返回
    raise NotImplementedError

In [ ]:
%%ipytest -qq

def test_build_awq_recipe_two_stages():
    recipe = build_awq_recipe()
    assert isinstance(recipe, list) and len(recipe) == 2
    # 第一段是 AWQModifier
    from llmcompressor.modifiers.transform.awq import AWQModifier
    assert isinstance(recipe[0], AWQModifier)
    # 第二段是 QuantizationModifier，scheme=W4A16_ASYM，ignore 含 lm_head
    assert recipe[1].scheme == "W4A16_ASYM"
    assert "lm_head" in list(recipe[1].ignore)
    assert isinstance(recipe[1].ignore, list)

def test_build_awq_recipe_custom_ignore_and_scheme():
    recipe = build_awq_recipe(ignore=("lm_head", "re:visual"), scheme="W4A16_ASYM")
    assert recipe[1].ignore == ["lm_head", "re:visual"]
    assert recipe[1].scheme == "W4A16_ASYM"

def test_build_calibration_dataset_from_raw_texts():
    # 免网络：直接传文本列表
    class FakeTok:
        model_max_length = 99
    texts = ["hello world " * 10, "another sample text", "third one"]
    ds = build_calibration_dataset(FakeTok(), n_samples=2, seq_len=64, raw_texts=texts)
    assert "text" in ds.column_names
    assert len(ds) == 2

def test_awq_config_summary_on_fake_w4a16():
    fake = {
        "quant_method": "compressed-tensors",
        "ignore": ["lm_head"],
        "config_groups": {"group_0": {
            "targets": ["Linear"],
            "weights": {"num_bits": 4, "symmetric": False, "group_size": 128, "strategy": "group"},
            "input_activations": None,
        }},
    }
    s = awq_config_summary(fake)
    assert s["quant_method"] == "compressed-tensors"
    assert s["weights_num_bits"] == 4
    assert s["weights_symmetric"] is False
    assert s["weights_group_size"] == 128
    assert s["weights_strategy"] == "group"
    assert s["has_input_activations"] is False
    assert s["targets"] == ["Linear"]

## L2：tiny 模型验证（CPU/GPU 秒~十秒级）

用 tiny Qwen2（hidden 须能被 group_size 128 整除，故 hidden=128）真跑 AWQ 两段式。
注意 AWQ 的 grid search 比 FP8 慢，所以 tiny 用最小规模（2 层、少量样本）。

In [ ]:
from transformers import Qwen2Config, Qwen2ForCausalLM, PreTrainedTokenizerFast
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from datasets import Dataset

def make_tiny_tokenizer(vocab_size=512):
    """构造词表对齐 tiny 模型的 word-level tokenizer（避免 token id 越界）。"""
    vocab = {str(i): i for i in range(vocab_size)}
    tk = Tokenizer(WordLevel(vocab=vocab, unk_token="0"))
    tk.pre_tokenizer = Whitespace()
    return PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="0", pad_token="0",
                                   eos_token="0", bos_token="0", model_max_length=64)

def make_tiny_model(vocab_size=512, hidden_size=128):
    cfg = Qwen2Config(num_hidden_layers=2, hidden_size=hidden_size,
                      intermediate_size=hidden_size * 2, num_attention_heads=4,
                      num_key_value_heads=2, vocab_size=vocab_size, tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()

tiny = make_tiny_model()
tiny_tok = make_tiny_tokenizer()
device = "cuda" if torch.cuda.is_available() else "cpu"
tiny.to(device)
print("tiny AWQ 模型就绪:", sum(p.numel() for p in tiny.parameters()), "params")

# 给 tiny 模型一份微型校准文本（词表内 token "0".."49"）
calib_texts = [" ".join(str(i % 50) for i in range(60))] * 8

# 真跑 AWQ（tiny 规模，group search 极小）
tiny_out = OUT_ROOT / "tiny-awq"
run_awq_quantize(tiny, tiny_tok, calib_texts, tiny_out, n_samples=8, seq_len=64)

qc = json.loads((tiny_out / "config.json").read_text())["quantization_config"]
summary = awq_config_summary(qc)
print("AWQ 产物摘要:", summary)
assert summary["weights_num_bits"] == 4
assert summary["weights_symmetric"] is False
assert summary["has_input_activations"] is False
print("L2 PASS：tiny AWQ 流水线跑通，产物为 W4A16")

## L3：H200 执行（真 Qwen2.5-0.5B 再 7B）

GPU 守卫：无 GPU 自动跳过。有 GPU 则先 0.5B 验证，再 7B 出可部署 W4A16 产物。

In [ ]:
if torch.cuda.is_available():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    # 7B 的 tokenizer 同时用于 0.5B 和 7B（同族 Qwen2.5 词表一致）
    tok = AutoTokenizer.from_pretrained(MODEL_DIR)

    # 先 0.5B 快验
    m05b = AutoModelForCausalLM.from_pretrained(TINY_MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_05b = OUT_ROOT / "qwen05b-awq"
    run_awq_quantize(m05b, tok, None, out_05b, n_samples=128, seq_len=512)
    print("0.5B AWQ done ->", out_05b)
    del m05b; torch.cuda.empty_cache()

    # 再 7B（出可部署产物；AWQ 省 token，256 样本足够）
    m7b = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype="auto", device_map="auto")
    out_7b = OUT_ROOT / "qwen7b-awq"
    run_awq_quantize(m7b, tok, None, out_7b, n_samples=256, seq_len=2048)
    print("7B AWQ done ->", out_7b)
    del m7b; torch.cuda.empty_cache()
else:
    print("跳过：无 GPU（CPU 环境只跑 L1/L2）。")

## 产物检查

打印 7B AWQ 产物的 `quantization_config`，并与 FP16 / FP8（若 Step 1 已跑）对比显存。

In [ ]:
def report_awq_artifact(out_dir):
    out_dir = pathlib.Path(out_dir)
    if not out_dir.exists():
        print(f"(跳过：{out_dir} 不存在，可能 L3 未跑)")
        return None
    cfg = json.loads((out_dir / "config.json").read_text())
    qc = cfg["quantization_config"]
    s = awq_config_summary(qc)
    total = sum(f.stat().st_size for f in out_dir.glob("*.safetensors"))
    print(f"== {out_dir.name} ==")
    print(f"  weights: {s['weights_num_bits']}-bit sym={s['weights_symmetric']} "
          f"group={s['weights_group_size']} strategy={s['weights_strategy']}")
    print(f"  input_activations 量化: {s['has_input_activations']}（W4A16 = 不量化激活）")
    print(f"  safetensors 总大小: {total/1e9:.2f} GB")
    return total

sizes = {"AWQ 7B": report_awq_artifact(OUT_ROOT / "qwen7b-awq")}
fp16 = sum(f.stat().st_size for f in MODEL_DIR.glob("*.safetensors"))
print(f"\n== 原始 FP16 7B: {fp16/1e9:.2f} GB ==")
if sizes["AWQ 7B"]:
    print(f"== AWQ/FP16 磁盘比: {sizes['AWQ 7B']/fp16:.2%}（W4A16 理论约 25%~30%）==")
fp8 = OUT_ROOT / "qwen7b-fp8"
if fp8.exists():
    fp8_size = sum(f.stat().st_size for f in fp8.glob("*.safetensors"))
    print(f"== 对比 Step1 FP8: {fp8_size/1e9:.2f} GB ==")